# Journal → JSON round-trip

Every plotlet `Chart` and `Layout` is a recorded *journal* — a list of every
artist, frame, and state-method call you made. `pt.to_json(node)` emits that
journal as a JSON-compatible dict; `pt.from_json(blob)` builds a fresh object
that renders to the same SVG.

This makes plots:

- **Storable** as plain text alongside the code that produced them
- **Diffable** by semantic content, not pixel output
- **AI-readable** without running Python — the JSON describes intent (`scatter`,
  `share_y`, `attach_left`) directly


## 1. Build a plot

In [1]:
import json
import plotlet as pt

left = pt.chart(data_width=240, data_height=160, title="raw")
left.scatter(
    data={"x": [1, 2, 3, 4, 5, 6],
          "y": [0.8, 1.4, 2.1, 1.7, 2.6, 3.0]},
    x="x", y="y", color="#1f77b4",
)
left.xlabel("dose")
left.ylabel("response")

right = pt.chart(data_width=240, data_height=160, title="log-y")
right.scatter(
    data={"x": [1, 2, 3, 4, 5, 6],
          "y": [0.8, 1.4, 2.1, 1.7, 2.6, 3.0]},
    x="x", y="y", color="#d62728",
)
right.yscale("log")
right.xlabel("dose")

fig = (left | right).share_y("all").gap(8)
fig

## 2. Serialize to JSON

`pt.to_json` walks the reachable graph, assigns each node a stable id, and
emits a flat list of records.


In [2]:
blob = pt.to_json(fig)
print(json.dumps(blob, indent=2))

{
  "version": 1,
  "root_nid": 1,
  "entries": [
    {
      "op": "new_chart",
      "nid": 2,
      "args": [],
      "kwargs": {
        "data": null,
        "data_width": 240,
        "data_height": 160,
        "margin": {
          "top": 0,
          "right": 0,
          "bottom": 0,
          "left": 0
        }
      }
    },
    {
      "op": "title",
      "nid": 2,
      "args": [
        "raw"
      ],
      "kwargs": {}
    },
    {
      "op": "scatter",
      "nid": 2,
      "args": [],
      "kwargs": {
        "data": {
          "x": [
            1,
            2,
            3,
            4,
            5,
            6
          ],
          "y": [
            0.8,
            1.4,
            2.1,
            1.7,
            2.6,
            3.0
          ]
        },
        "x": "x",
        "y": "y",
        "color": "#1f77b4"
      }
    },
    {
      "op": "xlabel",
      "nid": 2,
      "args": [
        "dose"
      ],
      "kwargs": {}
    },
    {

Notes on the shape:

- Each Chart records its construction-time state (`data_width`, `data_height`,
  `data`, `aes`, `margin`) plus its `calls` list.
- Each Layout records its `kind`, `children` (as `{"$ref": "..."}` pointers
  into the node table), and its own `calls` (here: `share_y`, `gap`).
- Args and kwargs hold the same values the user passed — the `scatter` entry
  reads like the source code.


## 3. Reconstruct from JSON

In [3]:
fig2 = pt.from_json(blob)
fig2

<JournalNode nid=1 entries=13>

## 4. Confirm identical

Both renders compare byte-identical (modulo the clip-path id that includes
Python's `id(obj)` for the panel state — that one identifier is volatile by
construction; everything else is reproducible).


In [4]:
import re
_VOLATILE = re.compile(r' id="pc[0-9a-f]+"| clip-path="url\(#pc[0-9a-f]+\)"')
def norm(svg): return _VOLATILE.sub("", svg)

a = norm(fig.to_svg())
b = norm(fig2.to_svg())
print(f"identical: {a == b}   ({len(a)} bytes each)")

identical: True   (20681 bytes each)
